# MediDiagnose — Presentation Figures & Results

Generates **slide-ready figures** for your defense / presentation, aligned with **Section VI (Table 2)** of the submitted paper.

**Outputs:** `documentations/figures/presentation/*.png`

**Prerequisites:**
- Trained model bundle in `backend/symptom_disease_model/` (run `train_on_colab.ipynb` first)
- `evaluation_results.json` written by training
- Optional: existing XAI figures in `documentations/figures/`

Run all cells top-to-bottom, then insert PNGs into PowerPoint / Canva.

In [ ]:
%pip install -q matplotlib seaborn pandas numpy

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
import seaborn as sns

ROOT = Path('..').resolve()
BACKEND = ROOT / 'backend'
MODEL_DIR = BACKEND / 'symptom_disease_model'
CSV_PATH = ROOT / 'DiseaseAndSymptoms.csv'
FIG_DIR = ROOT / 'documentations' / 'figures' / 'presentation'
XAI_DIR = ROOT / 'documentations' / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Presentation theme
PALETTE = {
    'primary': '#2563eb',
    'accent': '#0d9488',
    'success': '#16a34a',
    'warn': '#ea580c',
    'muted': '#64748b',
    'bg': '#f8fafc',
}
sns.set_theme(style='whitegrid', context='talk', font_scale=0.95)
plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 200,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

def save_fig(name: str, fig=None):
    path = FIG_DIR / name
    target = fig or plt.gcf()
    target.savefig(path, bbox_inches='tight', facecolor='white')
    print(f'Saved {path.relative_to(ROOT)}')
    return path

print(f'Project root: {ROOT}')
print(f'Figure output: {FIG_DIR}')

## 1. Load published results (Table 2)

In [ ]:
eval_path = MODEL_DIR / 'evaluation_results.json'
meta_path = MODEL_DIR / 'training_meta.json'

if eval_path.exists():
    with open(eval_path, encoding='utf-8') as f:
        eval_data = json.load(f)
    table2 = eval_data.get('test', {})
else:
    print('Warning: evaluation_results.json not found — using paper Table 2 defaults')
    table2 = {
        'top1_accuracy': 0.814,
        'top3_accuracy': 0.937,
        'macro_f1': 0.796,
        'weighted_f1': 0.812,
        'mean_confidence_correct': 0.873,
        'mean_confidence_incorrect': 0.531,
        'test_samples': 492,
    }

meta = {}
if meta_path.exists():
    with open(meta_path, encoding='utf-8') as f:
        meta = json.load(f)

rows = [
    ('Top-1 Accuracy', table2['top1_accuracy'], '%'),
    ('Top-3 Accuracy', table2['top3_accuracy'], '%'),
    ('Macro F1', table2['macro_f1'], ''),
    ('Weighted F1', table2['weighted_f1'], ''),
    ('Mean confidence (correct)', table2['mean_confidence_correct'], ''),
    ('Mean confidence (incorrect)', table2['mean_confidence_incorrect'], ''),
]
df_table2 = pd.DataFrame(rows, columns=['Metric', 'Value', 'Unit'])
df_table2['Display'] = df_table2.apply(
    lambda r: f"{r['Value']*100:.1f}%" if r['Unit'] == '%' else f"{r['Value']:.3f}", axis=1
)
display(df_table2[['Metric', 'Display']])
print(f"Test samples: {table2.get('test_samples', 492)}")

## 2. Table 2 — classification performance (bar chart)

In [ ]:
metrics = ['Top-1', 'Top-3', 'Macro F1', 'Weighted F1']
values = [
    table2['top1_accuracy'] * 100,
    table2['top3_accuracy'] * 100,
    table2['macro_f1'] * 100,
    table2['weighted_f1'] * 100,
]
colors = [PALETTE['primary'], PALETTE['accent'], PALETTE['success'], '#7c3aed']

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(metrics, values, color=colors, edgecolor='white', linewidth=1.5, width=0.62)
ax.set_ylim(0, 105)
ax.set_ylabel('Score (%)')
ax.set_title('Table 2 — NLP Classification Performance (Test set, n=492)', fontweight='bold', pad=16)
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width() / 2, val + 1.2, f'{val:.1f}%',
            ha='center', va='bottom', fontsize=13, fontweight='600')
ax.axhline(80, color=PALETTE['muted'], linestyle='--', linewidth=1, alpha=0.5)
fig.text(0.5, 0.02, 'Bio_ClinicalBERT · 80/10/10 row split · partial-symptom test protocol (Section VI)',
         ha='center', fontsize=10, color=PALETTE['muted'])
save_fig('fig_table2_performance.png', fig)
plt.show()

## 3. Confidence calibration (correct vs incorrect predictions)

In [ ]:
conf_ok = table2['mean_confidence_correct']
conf_bad = table2['mean_confidence_incorrect']
gap = conf_ok - conf_bad

fig, ax = plt.subplots(figsize=(8, 5.5))
bars = ax.bar(['Correct predictions', 'Incorrect predictions'], [conf_ok, conf_bad],
              color=[PALETTE['success'], PALETTE['warn']], width=0.55, edgecolor='white')
ax.set_ylim(0, 1.05)
ax.set_ylabel('Mean softmax confidence')
ax.set_title('Confidence gap supports clinical triage', fontweight='bold', pad=14)
for bar, val in zip(bars, [conf_ok, conf_bad]):
    ax.text(bar.get_x() + bar.get_width() / 2, val + 0.03, f'{val:.3f}',
            ha='center', fontweight='600')
ax.annotate('', xy=(1, conf_bad), xytext=(0, conf_ok),
            arrowprops=dict(arrowstyle='<->', color=PALETTE['muted'], lw=1.5))
ax.text(0.5, (conf_ok + conf_bad) / 2 + 0.06, f'Δ = {gap:.2f}',
        ha='center', fontsize=12, color=PALETTE['muted'])
save_fig('fig_confidence_gap.png', fig)
plt.show()

## 4. Model comparison (Section VI baselines)

In [ ]:
comparison = pd.DataFrame({
    'Model': ['TF-IDF + SVM', 'GRU', 'Bio_ClinicalBERT\n(MediDiagnose)'],
    'Top-1 (%)': [68.2, 73.5, 81.4],
    'Top-3 (%)': [81.1, 86.9, 93.7],
})
display(comparison)

x = np.arange(len(comparison))
width = 0.35
fig, ax = plt.subplots(figsize=(10, 6))
ax.bar(x - width/2, comparison['Top-1 (%)'], width, label='Top-1', color=PALETTE['primary'])
ax.bar(x + width/2, comparison['Top-3 (%)'], width, label='Top-3', color=PALETTE['accent'])
ax.set_xticks(x)
ax.set_xticklabels(comparison['Model'])
ax.set_ylabel('Accuracy (%)')
ax.set_ylim(60, 100)
ax.set_title('Algorithm comparison on symptom–disease benchmark', fontweight='bold', pad=14)
ax.legend(frameon=True, loc='lower right')
save_fig('fig_model_comparison.png', fig)
plt.show()

## 5. Dataset & train/val/test split

In [ ]:
df = pd.read_csv(CSV_PATH)
df.columns = df.columns.str.strip()
symptom_cols = [c for c in df.columns if c.startswith('Symptom')]
n_rows = len(df)
n_diseases = df['Disease'].nunique()
symptoms = set()
for col in symptom_cols:
    symptoms.update(df[col].dropna().astype(str).str.strip().str.lower())
symptoms.discard('')
symptoms.discard('nan')
n_symptoms = len(symptoms)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Split pie
split_labels = ['Train (80%)', 'Validation (10%)', 'Test (10%)']
split_sizes = [int(n_rows * 0.8), int(n_rows * 0.1), n_rows - int(n_rows * 0.8) - int(n_rows * 0.1)]
axes[0].pie(split_sizes, labels=split_labels, autopct='%1.1f%%', startangle=90,
            colors=[PALETTE['primary'], PALETTE['accent'], PALETTE['success']],
            textprops={'fontsize': 11})
axes[0].set_title(f'Data split ({n_rows:,} rows)', fontweight='bold')

# Key stats
stats = [
    ('CSV rows', f'{n_rows:,}'),
    ('Disease classes', str(n_diseases)),
    ('Unique symptoms', str(n_symptoms)),
    ('Test set (Table 2)', str(table2.get('test_samples', 492))),
]
axes[1].axis('off')
y = 0.85
axes[1].text(0.05, 0.95, 'Dataset summary', fontsize=16, fontweight='bold', transform=axes[1].transAxes)
for label, val in stats:
    axes[1].text(0.08, y, label, fontsize=13, color=PALETTE['muted'], transform=axes[1].transAxes)
    axes[1].text(0.72, y, val, fontsize=14, fontweight='600', transform=axes[1].transAxes)
    y -= 0.18
axes[1].add_patch(mpatches.FancyBboxPatch((0.02, 0.08), 0.96, 0.88, boxstyle='round,pad=0.02',
                                           linewidth=1, edgecolor='#e2e8f0', facecolor=PALETTE['bg'],
                                           transform=axes[1].transAxes))

plt.tight_layout()
save_fig('fig_dataset_split.png')
plt.show()

## 6. Training convergence (from Trainer logs or last run)

In [ ]:
trainer_state = None
checkpoints = MODEL_DIR / 'checkpoints'
for candidate in [MODEL_DIR / 'trainer_state.json']:
    if candidate.exists():
        trainer_state = json.loads(candidate.read_text(encoding='utf-8'))
if checkpoints.exists():
    for cp in sorted(checkpoints.iterdir()):
        ts = cp / 'trainer_state.json'
        if ts.exists():
            trainer_state = json.loads(ts.read_text(encoding='utf-8'))

if trainer_state and 'log_history' in trainer_state:
    logs = trainer_state['log_history']
    train_steps = [l['step'] for l in logs if 'loss' in l and 'eval_loss' not in l]
    train_loss = [l['loss'] for l in logs if 'loss' in l and 'eval_loss' not in l]
    eval_steps = [l['step'] for l in logs if 'eval_loss' in l]
    eval_loss = [l['eval_loss'] for l in logs if 'eval_loss' in l]
    eval_acc = [l.get('eval_accuracy', 0) for l in logs if 'eval_loss' in l]
    source = 'trainer_state.json'
else:
    # Fallback: representative curve from last Colab training run
    epochs = list(range(1, 11))
    train_loss = [3.59, 2.37, 1.15, 0.63, 0.34, 0.19, 0.12, 0.09, 0.08, 0.07]
    eval_loss = [3.34, 1.85, 1.02, 0.56, 0.34, 0.20, 0.14, 0.11, 0.09, 0.09]
    eval_acc = [0.24, 0.85, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
    train_steps = epochs
    eval_steps = epochs
    source = 'last documented training run (fallback)'

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].plot(train_steps, train_loss, 'o-', color=PALETTE['primary'], label='Train loss', linewidth=2)
axes[0].plot(eval_steps, eval_loss, 's-', color=PALETTE['warn'], label='Val loss', linewidth=2)
axes[0].set_xlabel('Epoch / step')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training & validation loss', fontweight='bold')
axes[0].legend()

axes[1].plot(eval_steps, eval_acc, 'o-', color=PALETTE['success'], linewidth=2.5)
axes[1].set_xlabel('Epoch / step')
axes[1].set_ylabel('Validation accuracy')
axes[1].set_ylim(0, 1.05)
axes[1].set_title('Validation accuracy per epoch', fontweight='bold')

fig.suptitle(f'Fine-tuning Bio_ClinicalBERT — source: {source}', y=1.02, fontsize=11, color=PALETTE['muted'])
plt.tight_layout()
save_fig('fig_training_curves.png', fig)
plt.show()

## 7. Explainability (SHAP + LIME) — existing case studies

In [ ]:
from IPython.display import Image, display

cases = [
    ('Fungal infection', 'fig_fungal_infection_shap_lime_combined.png'),
    ('Malaria', 'fig_malaria_shap_lime_combined.png'),
    ('Diabetes', 'fig_diabetes_shap_lime_combined.png'),
]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, (title, fname) in zip(axes, cases):
    path = XAI_DIR / fname
    if path.exists():
        img = plt.imread(path)
        ax.imshow(img)
        ax.set_title(title, fontweight='bold')
    else:
        ax.text(0.5, 0.5, f'Missing\n{fname}\n\nRun: python backend/generate_xai_figures.py',
                ha='center', va='center', transform=ax.transAxes, fontsize=10)
        ax.set_title(title)
    ax.axis('off')

fig.suptitle('SHAP + LIME word-importance (Section IV)', fontweight='bold', y=1.02)
plt.tight_layout()
save_fig('fig_xai_gallery.png', fig)
plt.show()

## 8. Physician trust study (Section VI)

In [ ]:
conditions = ['Without SHAP', 'With SHAP']
likert = [2.9, 4.1]

fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.bar(conditions, likert, color=[PALETTE['muted'], PALETTE['success']], width=0.5)
ax.set_ylim(0, 5.2)
ax.set_ylabel('Mean agreement (5-point Likert)')
ax.set_title('Physician agreement with AI prediction\n(n=10 physicians, 50 cases each)', fontweight='bold')
for bar, val in zip(bars, likert):
    ax.text(bar.get_x() + bar.get_width() / 2, val + 0.08, f'{val:.1f}',
            ha='center', fontweight='600', fontsize=14)
ax.axhline(3, color='#cbd5e1', linestyle='--', linewidth=1)
save_fig('fig_physician_trust.png', fig)
plt.show()

## 9. System latency (Section VI)

In [ ]:
latency = pd.DataFrame({
    'Scenario': ['Cold start\n(first request)', 'Warm inference\n(cached model)', 'API (non-AI)'],
    'Seconds': [12.8, 3.2, 0.1],
})
display(latency)

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.barh(latency['Scenario'], latency['Seconds'],
               color=[PALETTE['warn'], PALETTE['primary'], PALETTE['accent']])
ax.set_xlabel('Latency (seconds)')
ax.set_title('End-to-end inference latency', fontweight='bold')
for bar, val in zip(bars, latency['Seconds']):
    ax.text(val + 0.3, bar.get_y() + bar.get_height() / 2, f'{val}s', va='center', fontweight='600')
save_fig('fig_system_latency.png', fig)
plt.show()

## 10. One-slide summary (all key numbers)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))
ax.axis('off')

lines = [
    'MediDiagnose — Key Results (Section VI)',
    '',
    f"• Top-1 accuracy: {table2['top1_accuracy']*100:.1f}%   |   Top-3: {table2['top3_accuracy']*100:.1f}%",
    f"• Macro F1: {table2['macro_f1']:.3f}   |   Weighted F1: {table2['weighted_f1']:.3f}",
    f"• Confidence: {table2['mean_confidence_correct']:.3f} (correct) vs {table2['mean_confidence_incorrect']:.3f} (incorrect)",
    f"• Dataset: {n_rows:,} rows · {n_diseases} diseases · {n_symptoms} symptoms · test n={table2.get('test_samples', 492)}",
    '• Model: Bio_ClinicalBERT fine-tuned · AdamW 2e-5 · batch 16 · 10 epochs',
    '• XAI: SHAP + LIME surfaced in physician UI',
    '• Governance: mandatory doctor lock-and-review before patient delivery',
    '• Physician study: Likert 2.9 → 4.1 with SHAP explanations (p < 0.05)',
]
y = 0.92
for i, line in enumerate(lines):
    weight = 'bold' if i == 0 else 'normal'
    size = 18 if i == 0 else 13
    color = PALETTE['primary'] if i == 0 else '#1e293b'
    ax.text(0.06, y, line, fontsize=size, fontweight=weight, color=color, transform=ax.transAxes)
    y -= 0.085 if line else 0.04

ax.add_patch(mpatches.FancyBboxPatch((0.03, 0.05), 0.94, 0.9, boxstyle='round,pad=0.02',
                                       linewidth=2, edgecolor=PALETTE['primary'], facecolor='white',
                                       transform=ax.transAxes, alpha=0.9))
save_fig('fig_summary_slide.png', fig)
plt.show()

## Exported files

| File | Use in presentation |
|------|---------------------|
| `fig_table2_performance.png` | Main results slide (Table 2) |
| `fig_confidence_gap.png` | Confidence / calibration slide |
| `fig_model_comparison.png` | BERT vs baselines |
| `fig_dataset_split.png` | Methodology / dataset slide |
| `fig_training_curves.png` | Training convergence |
| `fig_xai_gallery.png` | Explainability demo |
| `fig_physician_trust.png` | User study results |
| `fig_system_latency.png` | System performance |
| `fig_summary_slide.png` | Closing / overview slide |

**Tip:** Re-run after updating `evaluation_results.json` from Colab so Table 2 stays in sync with the paper.